# Rarity-driven sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — IDF over the pool, irregular-term list | terms are scriptural vocabulary, not tokenizer debris |
| Step 3 | `leakage` — near-duplicate audit, quarantine | flag count small enough that quarantining leaves the pool intact |
| Step 2 | `sparse_select` — greedy coverage dry run | the channel fires often enough, and is less redundant than dense |


In [ ]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml

---
## Step 1 — the rarity list

IDF over the source side of `data/splits/train.jsonl`, keeping the rarest terms.

In [ ]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

In [ ]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
print(f"{rarity['n_terms']} pool terms -> {rarity['n_irregular']} irregular "
      f"(requested {rarity['config']['top_frac']:.0%}, realized {rarity['realized_frac']:.1%}, "
      f"df <= {rarity['cutoff_df']})")
print('pool df histogram     :', rarity['df_histogram']['pool'])
print('irregular df histogram:', rarity['df_histogram']['irregular'])

sample = pd.read_csv('results/rarity_train_sample50.tsv', sep='\t')
sample['example'] = sample['example'].str.slice(0, 60)
sample

### Normalization check — ZWNJ

In [ ]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

---
## Step 3 — leakage audit

In [ ]:
!python3 manage.py leakage --config configs/sparse_retrieval.yaml --split val test --write-quarantine

In [ ]:
for split in ('val', 'test'):
    leak = json.load(open(f'results/leakage_{split}.json'))
    print(f"{split}: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
          f"{leak['n_pool_rows_flagged']} pool rows implicated")
    print('  max-cos histogram:', leak['max_cos_histogram'])

worst = json.load(open('results/leakage_val.json'))['flags'][:10]
pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in worst
])

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

---
## Step 2 — the sparse channel

In [ ]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

In [ ]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('irregular terms/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('coverage (routed) :', sel['coverage']['mean'])
print('intra-set cosine  :', sel['intra_set_similarity'])

### Threshold sweep

`min_query_terms` is the one knob that decides whether the channel exists at all. Selection
is cheap once the index is loaded, so sweep it rather than arguing about it. Each run
writes its own report, leaving the configured run's above intact.

In [ ]:
for thr in (1, 2, 3, 4):
    print(f'--- min_query_terms={thr}')
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --min_query_terms {thr} --out results/sparse_sweep_val_t{thr}.json 2>&1 | grep -E 'routes|intra-set'

### Worked examples

The trace behind three routed queries: which irregular terms the query carried, how much
of that rarity the selected set covered, and which exemplars were chosen.

In [ ]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| terms', ex['trace']['query_terms'],
          '| coverage', ex['trace']['coverage'])
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()